# Session 10: Enterprise Security, Multi-Agent Patterns & Guardrails

Advanced agentic patterns: reflection/critique loops, compliance checking, human-in-the-loop gates, and enterprise security hardening.

## 1. Reflection / Critique Pattern

In [ ]:
import asyncio
from src.agents.critic import CriticAgent
from src.agents.compliance import ComplianceAgent
from src.agents.risk_assessor import RiskAssessorAgent
from src.agents.portfolio_manager import PortfolioManagerAgent

critic = CriticAgent()
compliance = ComplianceAgent()
risk = RiskAssessorAgent()
portfolio = PortfolioManagerAgent()

print("All enterprise agents loaded")
print(f"Critic max iterations: {critic.max_iterations}")

### How the Reflection Pattern Works

The CriticAgent implements a reflection loop:
1. Analyst produces analysis → routed to `critic_analysis` node
2. CriticAgent reviews for gaps, biases, unsupported claims
3. If iteration < max_iterations (2), routes BACK to analyst for improvement
4. After max iterations or approval, proceeds to portfolio_manager

In [ ]:
# Demonstrate the critique cycle
sample_analysis = """
AAPL is a strong buy with excellent growth prospects.
The company continues to innovate. Revenue is growing.
Management is executing well.
"""

critique = await critic.critique_analysis(
    "Should I invest in Apple?",
    sample_analysis,
    {"servers": ["AAPL"], "mcp_results": {"market_data": "AAPL: $225"}}
)

print("=== Critique Results ===")
print(critique["critique"][:500])
print("\n(Full critique would identify missing metrics, lack of bear case, etc.)")

## 2. Compliance Checking

In [ ]:
# Demonstrate compliance checking
sample_report = """
Apple Inc. (AAPL) — Investment Analysis

Based on our analysis, Apple shows strong fundamentals with a P/E of 35x.
Revenue grew 8% YoY to $385B. We recommend BUY with a $250 price target.

Disclaimer: Past performance is not indicative of future results.
This is not financial advice. Consult your financial advisor.
"""

result = await compliance.check(sample_report)

print(f"Compliant: {result['compliant']}")
print(f"Issues found: {len(result['compliance_issues'])}")
for issue in result['compliance_issues']:
    print(f"  [{issue['severity']}] {issue['detail']}")

### Compliance Rules Enforced

| Rule | Severity | What it checks |
|------|----------|----------------|
| Regulatory keywords | High | "guarantee", "risk-free", "no risk", "certain", "sure thing" |
| Insider trading | High | "insider", "material nonpublic" |
| Guaranteed returns | High | "guaranteed return", "promised", "assured" |
| Past performance disclaimer | Medium | "past performance" must be present |
| Not financial advice | Medium | "not financial advice" required |
| Consult advisor | Medium | "consult your advisor" required |

## 3. Human-in-the-Loop (HITL)

In [ ]:
# HITL gate flow
hitl_flow = """
Graph Node: human_review
────────────────────
Trigger: After compliance check, if ENABLE_HUMAN_REVIEW=true
Gate: Waits for human_approved=True/False
  → If approved: proceeds to guard_output → END
  → If rejected: loops back to writer for revision
Use case: High-value reports, first-time queries, risk threshold exceeded
"""

print(hitl_flow)

### HITL Configuration

```python
# In creation graph
graph = create_graph(enable_human_review=True)
```

Set via environment variable:
```
ENABLE_HUMAN_REVIEW=true
```

## 4. Portfolio Manager Agent

In [ ]:
# Portfolio manager demo
allocation_models = {
    "conservative": {"equities": "30%", "bonds": "50%", "cash": "15%", "alternatives": "5%"},
    "moderate": {"equities": "55%", "bonds": "30%", "cash": "10%", "alternatives": "5%"},
    "aggressive": {"equities": "80%", "bonds": "10%", "cash": "5%", "alternatives": "5%"},
}

for profile, alloc in allocation_models.items():
    print(f"\n{profile.upper():15} ", end="")
    for asset, pct in alloc.items():
        print(f"{asset}: {pct}", end=" | ")
print()

## 5. Risk Assessment Agent

In [ ]:
# Risk assessment dimensions
risk_dimensions = """
RISK ASSESSMENT FRAMEWORK
══════════════════════════

1. Overall Risk Score (0.0-1.0)
   → Weighted composite of all factors

2. Key Risk Factors
   ├── Market Risk (beta, volatility)
   ├── Sector Risk (concentration, regulation)
   ├── Company-Specific (management, competitive)
   └── Macro Risk (rates, inflation, geopolitics)

3. VaR Estimate (95%, 1-month)
   → Maximum expected loss at 95% confidence

4. Maximum Drawdown Estimate
   → Peak-to-trough decline in stress scenario

5. Risk/Reward Ratio
   → Upside potential / downside risk

6. Mitigation Strategies
   → Hedging, diversification, position sizing
"""

print(risk_dimensions)

## 6. Updated Agent Graph Architecture

In [ ]:
new_graph = """
START → guard_input
  │
  ├─ [BLOCKED] → END
  └─ [PASS] → supervisor
                │
                ▼
            researcher
                │
                ▼
            analyst
                │
                ▼
            critic_analysis ←─────────────┐
                │                          │
          ┌─────┴─────┐                   │
          │ [PASS]    │ [REVISE] ──────────┘
          ▼           │
    portfolio_manager │
          │           │
          ▼           │
    risk_assessor     │
          │           │
          ▼           │
     writer           │
          │           │
          ▼           │
    critic_report     │
          │           │
          ▼           │
    compliance        │
          │           │
     ┌────┴────┐      │
     │ [PASS]  │ [FAIL] → (loops to writer)
     ▼         │
  human_review │
     │         │
 ┌───┴───┐     │
 │ [YES] │ [NO] ──────────→ writer (revision)
 ▼       │
guard_output
    │
    ▼
   END
"""

print(new_graph)

In [ ]:
print("Session 10 complete — enterprise security: reflection pattern, compliance, HITL, risk assessment")